In [1]:
import numpy as np
import pandas as pd
import zipfile
import shutil
import csv
import pandas as pd
import re
import unicodedata
import time
import requests
import fitz  

from tqdm import tqdm
from pathlib import Path
from io import BytesIO

In [2]:
OUTDIR = Path("datos_dengue_historicos")
PDF_DIR = OUTDIR / "pdfs"
ZIP_DIR = OUTDIR / "zips"

PDF_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

PDF_URLS = {
    2020: "https://www.gob.mx/cms/uploads/attachment/file/674027/Datos_abiertos_historicos_etv_2020.pdf",
    2021: "https://www.gob.mx/cms/uploads/attachment/file/691591/Datos_abiertos_historicos_etv_2021.pdf",
    2022: "https://www.gob.mx/cms/uploads/attachment/file/788955/Datos_abiertos_historicos_etv_2022.pdf",
    2023: "https://www.gob.mx/cms/uploads/attachment/file/891410/Datos_abiertos_historicos_etv_2023.pdf",
    2024: "https://www.gob.mx/cms/uploads/attachment/file/965318/Datos_abiertos_historicos_etv_2024.pdf",
    2025: "https://www.gob.mx/cms/uploads/attachment/file/1047102/Datos_abiertos_historicos_etv_2025.pdf",
    2026: "https://www.gob.mx/cms/uploads/attachment/file/1082563/Datos_abiertos_historicos_etv_2026.pdf",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,"
              "application/pdf,*/*;q=0.8",
    "Referer": "https://www.gob.mx/",
}



# Funciones auxiliares


def get_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def download_file(session, url, dest, desc=None, overwrite=False):
    dest = Path(dest)

    if dest.exists() and not overwrite and dest.stat().st_size > 0:
        print(f"[SKIP] Ya existe: {dest}")
        return dest

    tmp = dest.with_suffix(dest.suffix + ".part")

    with session.get(url, stream=True, timeout=180, allow_redirects=True) as r:
        r.raise_for_status()

        total = int(r.headers.get("content-length", 0))

        with open(tmp, "wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=desc or dest.name
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 128):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    tmp.replace(dest)
    return dest


def extract_zip_links_from_pdf(pdf_path):
    """
    Extrae las ligas .zip embebidas en los textos 'ver' del PDF.
    """
    pdf_path = Path(pdf_path)
    zip_links = []

    doc = fitz.open(pdf_path)

    for page in doc:
        for link in page.get_links():
            uri = link.get("uri")
            if uri and ".zip" in uri.lower():
                zip_links.append(uri)

    doc.close()

    # Quitar duplicados conservando orden
    seen = set()
    unique_links = []

    for url in zip_links:
        if url not in seen:
            unique_links.append(url)
            seen.add(url)

    return unique_links


def infer_publication_date_from_zip(url):
    """
    Extrae fechas tipo:
    datos_abiertos_dengue_080526.zip -> 2026-05-08
    """
    m = re.search(r"datos_abiertos_dengue_(\d{6})\.zip", url)
    if not m:
        return ""

    ddmmaa = m.group(1)
    dd = ddmmaa[0:2]
    mm = ddmmaa[2:4]
    yy = ddmmaa[4:6]

    return f"20{yy}-{mm}-{dd}"


def check_zip_ok(path):
    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        with zipfile.ZipFile(path, "r") as z:
            bad_file = z.testzip()
        return bad_file is None
    except zipfile.BadZipFile:
        return False




def main():
    session = get_session()
    manifest_rows = []

    print("\n======================================================")
    print("1. Descargando PDFs históricos")
    print("======================================================")

    local_pdfs = {}

    for year, pdf_url in PDF_URLS.items():
        pdf_path = PDF_DIR / f"datos_abiertos_historicos_etv_{year}.pdf"

        print(f"\n[INFO] PDF {year}")
        try:
            download_file(
                session=session,
                url=pdf_url,
                dest=pdf_path,
                desc=f"PDF {year}",
                overwrite=False
            )
            local_pdfs[year] = pdf_path
        except Exception as e:
            print(f"[ERROR] No se pudo descargar el PDF {year}")
            print(f"        {e}")

    print("\n======================================================")
    print("2. Extrayendo enlaces ZIP desde los PDFs")
    print("======================================================")

    all_zip_links = {}

    for year, pdf_path in local_pdfs.items():
        try:
            links = extract_zip_links_from_pdf(pdf_path)
            all_zip_links[year] = links
            print(f"[OK] {year}: {len(links)} ZIPs encontrados")
        except Exception as e:
            print(f"[ERROR] No se pudieron extraer links del PDF {year}")
            print(f"        {e}")
            all_zip_links[year] = []

    print("\n======================================================")
    print("3. Descargando ZIPs")
    print("======================================================")

    for year, links in all_zip_links.items():
        year_dir = ZIP_DIR / str(year)
        year_dir.mkdir(parents=True, exist_ok=True)

        for url in links:
            filename = url.split("/")[-1]
            dest = year_dir / filename

            print(f"\n[INFO] Descargando {year}/{filename}")

            try:
                download_file(
                    session=session,
                    url=url,
                    dest=dest,
                    desc=f"{year}/{filename}",
                    overwrite=False
                )

                zip_ok = check_zip_ok(dest)

                manifest_rows.append({
                    "year": year,
                    "publication_date": infer_publication_date_from_zip(url),
                    "zip_filename": filename,
                    "zip_url": url,
                    "local_path": str(dest),
                    "zip_ok": zip_ok,
                })

                if zip_ok:
                    print(f"[OK] ZIP válido: {dest}")
                else:
                    print(f"[WARN] Descargado, pero no parece ZIP válido: {dest}")

                time.sleep(0.4)

            except Exception as e:
                print(f"[ERROR] No se pudo descargar: {url}")
                print(f"        {e}")

                manifest_rows.append({
                    "year": year,
                    "publication_date": infer_publication_date_from_zip(url),
                    "zip_filename": filename,
                    "zip_url": url,
                    "local_path": str(dest),
                    "zip_ok": False,
                })

    print("\n======================================================")
    print("4. Guardando manifiesto")
    print("======================================================")

    manifest_path = OUTDIR / "manifest_descargas_dengue.csv"

    with open(manifest_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "year",
                "publication_date",
                "zip_filename",
                "zip_url",
                "local_path",
                "zip_ok",
            ]
        )
        writer.writeheader()
        writer.writerows(manifest_rows)

    total_zips = len(manifest_rows)
    valid_zips = sum(row["zip_ok"] for row in manifest_rows)

    print("\n======================================================")
    print("Proceso terminado")
    print("======================================================")
    print(f"Carpeta de salida: {OUTDIR.resolve()}")
    print(f"Manifiesto:        {manifest_path.resolve()}")
    print(f"ZIPs encontrados:  {total_zips}")
    print(f"ZIPs válidos:      {valid_zips}")


if __name__ == "__main__":
    main()


1. Descargando PDFs históricos

[INFO] PDF 2020
[SKIP] Ya existe: datos_dengue_historicos/pdfs/datos_abiertos_historicos_etv_2020.pdf

[INFO] PDF 2021
[SKIP] Ya existe: datos_dengue_historicos/pdfs/datos_abiertos_historicos_etv_2021.pdf

[INFO] PDF 2022
[SKIP] Ya existe: datos_dengue_historicos/pdfs/datos_abiertos_historicos_etv_2022.pdf

[INFO] PDF 2023
[SKIP] Ya existe: datos_dengue_historicos/pdfs/datos_abiertos_historicos_etv_2023.pdf

[INFO] PDF 2024
[SKIP] Ya existe: datos_dengue_historicos/pdfs/datos_abiertos_historicos_etv_2024.pdf

[INFO] PDF 2025
[SKIP] Ya existe: datos_dengue_historicos/pdfs/datos_abiertos_historicos_etv_2025.pdf

[INFO] PDF 2026
[SKIP] Ya existe: datos_dengue_historicos/pdfs/datos_abiertos_historicos_etv_2026.pdf

2. Extrayendo enlaces ZIP desde los PDFs
[OK] 2020: 5 ZIPs encontrados
[OK] 2021: 52 ZIPs encontrados
[OK] 2022: 52 ZIPs encontrados
[OK] 2023: 50 ZIPs encontrados
[OK] 2024: 49 ZIPs encontrados
[OK] 2025: 51 ZIPs encontrados
[OK] 2026: 20 ZIPs e

In [3]:
# Configuración

BASE = Path("datos_dengue_historicos")



MANIFEST = BASE / "manifest_descargas_dengue.csv"

OUTDIR = BASE / "csv_unificado"
EXTRACT_DIR = BASE / "extraidos"

OUTDIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000

# Cambia a False si no quieres guardar físicamente los archivos extraídos
EXTRAER_ARCHIVOS = True

EXTENSIONES_DATOS = {".csv", ".txt"}

ENCODINGS = ["utf-8-sig", "latin1", "cp1252"]
SEPARADORES = [",", "|", ";", "\t"]

META_COLS = [
    "snapshot_year",
    "publication_date",
    "zip_filename",
    "source_member",
    "source_zip",
]



# Funciones auxiliares


def limpiar_nombre_columna(col):
    col = str(col).replace("\ufeff", "").strip()
    col = re.sub(r"\s+", "_", col)
    return col.upper()


def limpiar_columnas(df):
    df.columns = [limpiar_nombre_columna(c) for c in df.columns]
    return df


def resolver_zip_path(row):
    """
    Intenta localizar el ZIP usando local_path del manifest.
    Si no existe, reconstruye la ruta usando año y nombre del archivo.
    """
    p = Path(str(row["local_path"]))

    if p.exists():
        return p

    p2 = BASE / "zips" / str(int(row["year"])) / str(row["zip_filename"])

    if p2.exists():
        return p2

    return p


def miembros_de_datos(zip_path):
    """
    Devuelve los archivos .csv o .txt dentro del ZIP.
    """
    miembros = []

    with zipfile.ZipFile(zip_path, "r") as z:
        for m in z.namelist():
            if m.endswith("/"):
                continue

            if "__MACOSX" in m:
                continue

            ext = Path(m).suffix.lower()

            if ext in EXTENSIONES_DATOS:
                miembros.append(m)

    return miembros


def detectar_formato_csv(data):
    """
    Detecta encoding y separador probando varias combinaciones.
    """
    mejor = None

    for enc in ENCODINGS:
        for sep in SEPARADORES:
            try:
                muestra = pd.read_csv(
                    BytesIO(data),
                    encoding=enc,
                    sep=sep,
                    dtype=str,
                    nrows=20,
                    engine="python",
                    on_bad_lines="warn",
                )

                muestra = limpiar_columnas(muestra)

                if len(muestra.columns) > 1:
                    return enc, sep, list(muestra.columns)

                if mejor is None:
                    mejor = (enc, sep, list(muestra.columns))

            except Exception:
                continue

    if mejor is not None:
        return mejor

    raise ValueError("No se pudo detectar encoding/separador del archivo.")


def extraer_miembro_seguro(z, member, out_base):
    """
    Extrae un archivo del ZIP evitando rutas peligrosas.
    """
    out_base = out_base.resolve()
    destino = (out_base / member).resolve()

    if not str(destino).startswith(str(out_base)):
        print(f"[WARN] Ruta sospechosa omitida: {member}")
        return

    destino.parent.mkdir(parents=True, exist_ok=True)

    with z.open(member) as src, open(destino, "wb") as dst:
        shutil.copyfileobj(src, dst)


def escanear_archivos(df_manifest):
    """
    Primera pasada:
    identifica archivos internos y obtiene la unión de columnas.
    """
    file_infos = []
    columnas = []
    columnas_vistas = set()

    for _, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc="Escaneando ZIPs"):
        zip_path = resolver_zip_path(row)

        if not zip_path.exists():
            print(f"[WARN] ZIP no encontrado: {zip_path}")
            continue

        try:
            miembros = miembros_de_datos(zip_path)

            if len(miembros) == 0:
                print(f"[WARN] Sin CSV/TXT dentro de: {zip_path}")
                continue

            with zipfile.ZipFile(zip_path, "r") as z:
                for member in miembros:
                    data = z.read(member)

                    enc, sep, cols = detectar_formato_csv(data)

                    info = {
                        "year": int(row["year"]),
                        "publication_date": row["publication_date"],
                        "zip_filename": row["zip_filename"],
                        "zip_path": zip_path,
                        "member": member,
                        "encoding": enc,
                        "sep": sep,
                    }

                    file_infos.append(info)

                    for c in cols:
                        if c not in columnas_vistas:
                            columnas.append(c)
                            columnas_vistas.add(c)

        except Exception as e:
            print(f"[ERROR] No se pudo escanear {zip_path}")
            print(e)

    all_columns = META_COLS + columnas

    return file_infos, all_columns


def escribir_csv_unificado(file_infos, all_columns, output_csv, resumen_csv):
    """
    Segunda pasada:
    lee los archivos en chunks y escribe un CSV unificado.
    """
    output_csv = Path(output_csv)
    resumen_csv = Path(resumen_csv)

    if output_csv.exists():
        output_csv.unlink()

    resumen = []
    primera_vez = True

    for info in tqdm(file_infos, desc=f"Construyendo {output_csv.name}"):
        zip_path = info["zip_path"]
        member = info["member"]

        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                data = z.read(member)

                if EXTRAER_ARCHIVOS:
                    out_extract = EXTRACT_DIR / str(info["year"]) / Path(zip_path).stem
                    extraer_miembro_seguro(z, member, out_extract)

            reader = pd.read_csv(
                BytesIO(data),
                encoding=info["encoding"],
                sep=info["sep"],
                dtype=str,
                chunksize=CHUNKSIZE,
                engine="python",
                on_bad_lines="warn",
            )

            n_rows_file = 0

            for chunk in reader:
                chunk = limpiar_columnas(chunk)

                chunk["snapshot_year"] = info["year"]
                chunk["publication_date"] = info["publication_date"]
                chunk["zip_filename"] = info["zip_filename"]
                chunk["source_member"] = info["member"]
                chunk["source_zip"] = str(info["zip_path"])

                for col in all_columns:
                    if col not in chunk.columns:
                        chunk[col] = pd.NA

                chunk = chunk[all_columns]

                chunk.to_csv(
                    output_csv,
                    mode="a",
                    index=False,
                    header=primera_vez,
                    encoding="utf-8",
                    quoting=csv.QUOTE_MINIMAL,
                )

                primera_vez = False
                n_rows_file += len(chunk)

            resumen.append({
                "snapshot_year": info["year"],
                "publication_date": info["publication_date"],
                "zip_filename": info["zip_filename"],
                "source_member": info["member"],
                "source_zip": str(info["zip_path"]),
                "n_rows": n_rows_file,
            })

        except Exception as e:
            print(f"[ERROR] No se pudo procesar {zip_path} :: {member}")
            print(e)

            resumen.append({
                "snapshot_year": info["year"],
                "publication_date": info["publication_date"],
                "zip_filename": info["zip_filename"],
                "source_member": info["member"],
                "source_zip": str(info["zip_path"]),
                "n_rows": 0,
                "error": str(e),
            })

    pd.DataFrame(resumen).to_csv(resumen_csv, index=False, encoding="utf-8")

    print(f"\n[OK] CSV generado: {output_csv}")
    print(f"[OK] Resumen generado: {resumen_csv}")



# Flujo principal


def main():
    if not MANIFEST.exists():
        raise FileNotFoundError(f"No encontré el manifest: {MANIFEST}")

    manifest = pd.read_csv(MANIFEST)

    manifest_validos = manifest[manifest["zip_ok"] == True].copy()

    manifest_validos["publication_date_dt"] = pd.to_datetime(
        manifest_validos["publication_date"],
        errors="coerce",
    )

    print("\n======================================================")
    print("Archivos válidos encontrados")
    print("======================================================")
    print(manifest_validos.groupby("year").size())
    print(f"\nTotal ZIPs válidos: {len(manifest_validos)}")

    # ------------------------------------------------------------
    # A) CSV con todos los cortes semanales
    # ------------------------------------------------------------

    print("\n======================================================")
    print("A) Construyendo CSV con todos los cortes semanales")
    print("======================================================")

    file_infos_all, all_columns_all = escanear_archivos(manifest_validos)

    escribir_csv_unificado(
        file_infos=file_infos_all,
        all_columns=all_columns_all,
        output_csv=OUTDIR / "dengue_todos_los_cortes.csv",
        resumen_csv=OUTDIR / "resumen_todos_los_cortes.csv",
    )



    print("\n======================================================")
    print("B) Construyendo CSV con última publicación por año")
    print("======================================================")

    ultimos_por_anio = (
        manifest_validos
        .dropna(subset=["publication_date_dt"])
        .sort_values(["year", "publication_date_dt"])
        .groupby("year", as_index=False)
        .tail(1)
        .sort_values("year")
    )

    print("\nÚltimos cortes usados por año:")
    print(
        ultimos_por_anio[
            ["year", "publication_date", "zip_filename"]
        ].to_string(index=False)
    )

    file_infos_latest, all_columns_latest = escanear_archivos(ultimos_por_anio)

    escribir_csv_unificado(
        file_infos=file_infos_latest,
        all_columns=all_columns_latest,
        output_csv=OUTDIR / "dengue_ultima_publicacion_por_anio.csv",
        resumen_csv=OUTDIR / "resumen_ultima_publicacion_por_anio.csv",
    )

    print("\n======================================================")
    print("Proceso terminado")
    print("======================================================")
    print(f"Carpeta de CSVs:      {OUTDIR.resolve()}")
    print(f"Carpeta de extraídos: {EXTRACT_DIR.resolve()}")


if __name__ == "__main__":
    main()


Archivos válidos encontrados
year
2020     5
2021    50
2022    50
2023    49
2024    45
2025    48
2026    20
dtype: int64

Total ZIPs válidos: 267

A) Construyendo CSV con todos los cortes semanales


Escaneando ZIPs:   2%|▌                         | 6/267 [00:00<00:04, 58.63it/s]

[WARN] Sin CSV/TXT dentro de: datos_dengue_historicos/zips/2020/datos_abiertos_dengue_031220.zip


Escaneando ZIPs: 100%|████████████████████████| 267/267 [00:03<00:00, 84.07it/s]
Construyendo dengue_todos_los_cortes.csv: 100%|█| 266/266 [04:16<00:00,  1.04it/



[OK] CSV generado: datos_dengue_historicos/csv_unificado/dengue_todos_los_cortes.csv
[OK] Resumen generado: datos_dengue_historicos/csv_unificado/resumen_todos_los_cortes.csv

B) Construyendo CSV con última publicación por año

Últimos cortes usados por año:
 year publication_date                     zip_filename
 2020       2020-12-31 datos_abiertos_dengue_311220.zip
 2021       2021-12-30 datos_abiertos_dengue_301221.zip
 2022       2022-12-29 datos_abiertos_dengue_291222.zip
 2023       2023-12-27 datos_abiertos_dengue_271223.zip
 2024       2024-12-19 datos_abiertos_dengue_191224.zip
 2025       2025-12-26 datos_abiertos_dengue_261225.zip
 2026       2026-05-28 datos_abiertos_dengue_280526.zip


Escaneando ZIPs: 100%|████████████████████████████| 7/7 [00:00<00:00, 36.37it/s]
Construyendo dengue_ultima_publicacion_por_anio.csv: 100%|█| 7/7 [00:17<00:00,  


[OK] CSV generado: datos_dengue_historicos/csv_unificado/dengue_ultima_publicacion_por_anio.csv
[OK] Resumen generado: datos_dengue_historicos/csv_unificado/resumen_ultima_publicacion_por_anio.csv

Proceso terminado
Carpeta de CSVs:      /Users/juanalbertomartinez/Desktop/Dengue/datos_dengue_historicos/csv_unificado
Carpeta de extraídos: /Users/juanalbertomartinez/Desktop/Dengue/datos_dengue_historicos/extraidos


In [16]:
BASE = Path("datos_dengue_historicos")

MANIFEST = BASE / "manifest_descargas_dengue.csv"

# Archivos externos 
MUNICIPIOS_CSV = Path("municipio.csv")
ESTADOS_XLSX = Path("estados.xlsx")
COORDS_CSV = Path("coordenadas.csv")

OUTDIR = BASE / "csv_unificado"
OUTDIR.mkdir(parents=True, exist_ok=True)

OUTPUT = OUTDIR / "dengue_2020_2026.csv"

YEARS = list(range(2020, 2027))

DROP_MISSING_COORDS = True

In [17]:
def quitar_acentos(texto):
    if pd.isna(texto):
        return np.nan
    
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        c for c in texto 
        if unicodedata.category(c) != "Mn"
    )
    texto = re.sub(r"\s+", " ", texto)
    return texto


def normalizar_columnas(df):
    df = df.copy()
    df.columns = [
        quitar_acentos(c).replace(" ", "_")
        for c in df.columns
    ]
    return df


def limpiar_codigo_entidad(x):
    """
    Convierte claves de entidad a formato 01, 02, ..., 32.
    Si encuentra valores como 'USA', 'EXTRANJERO' o texto no numérico,
    devuelve NaN en lugar de romper el código.
    """
    if pd.isna(x):
        return np.nan
    
    x_num = pd.to_numeric(x, errors="coerce")
    
    if pd.isna(x_num):
        return np.nan
    
    x_int = int(x_num)
    
    if x_int < 1 or x_int > 32:
        return np.nan
    
    return str(x_int).zfill(2)


def limpiar_codigo_municipio(x):
    """
    Convierte claves de municipio a formato 001, 002, ...
    Si encuentra valores no numéricos, devuelve NaN.
    """
    if pd.isna(x):
        return np.nan
    
    x_num = pd.to_numeric(x, errors="coerce")
    
    if pd.isna(x_num):
        return np.nan
    
    x_int = int(x_num)
    
    if x_int < 0:
        return np.nan
    
    return str(x_int).zfill(3)



def normalizar_sexo(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip().upper()
    
    if x in ["1", "1.0", "MUJER", "FEMENINO", "F", "M"]:
        return "M"
    elif x in ["2", "2.0", "HOMBRE", "MASCULINO", "H"]:
        return "H"
    else:
        return np.nan


def grupo_edad(edad):
    if pd.isna(edad):
        return np.nan
    
    edad = float(edad)
    
    if edad <= 17:
        return "<18"
    elif edad <= 29:
        return "18 a 29 anios"
    elif edad <= 39:
        return "30 a 39 anios"
    elif edad <= 49:
        return "40 a 49 anios"
    elif edad <= 59:
        return "50 a 59 anios"
    else:
        return ">=60 anios"

In [18]:
ENTIDADES = {
    "01": "AGUASCALIENTES",
    "02": "BAJA CALIFORNIA",
    "03": "BAJA CALIFORNIA SUR",
    "04": "CAMPECHE",
    "05": "COAHUILA",
    "06": "COLIMA",
    "07": "CHIAPAS",
    "08": "CHIHUAHUA",
    "09": "DISTRITO FEDERAL",
    "10": "DURANGO",
    "11": "GUANAJUATO",
    "12": "GUERRERO",
    "13": "HIDALGO",
    "14": "JALISCO",
    "15": "MEXICO",
    "16": "MICHOACAN",
    "17": "MORELOS",
    "18": "NAYARIT",
    "19": "NUEVO LEON",
    "20": "OAXACA",
    "21": "PUEBLA",
    "22": "QUERETARO",
    "23": "QUINTANA ROO",
    "24": "SAN LUIS POTOSI",
    "25": "SINALOA",
    "26": "SONORA",
    "27": "TABASCO",
    "28": "TAMAULIPAS",
    "29": "TLAXCALA",
    "30": "VERACRUZ",
    "31": "YUCATAN",
    "32": "ZACATECAS",
}


def asignar_region(entidad_res):
    """
    Asigna región a partir de la clave de entidad.
    Acepta claves tipo '01', '1', 1, 1.0.
    Si recibe NaN o una clave no válida, devuelve NaN.
    """
    if pd.isna(entidad_res):
        return np.nan
    
    clave = pd.to_numeric(entidad_res, errors="coerce")
    
    if pd.isna(clave):
        return np.nan
    
    clave = int(clave)
    
    if clave in [2, 3, 25, 26]:
        return "Region 1"
    elif clave in [8, 10, 32]:
        return "Region 2"
    elif clave in [5, 19, 24, 28]:
        return "Region 3"
    elif clave in [6, 14, 16, 18]:
        return "Region 4"
    elif clave in [7, 12, 20, 30]:
        return "Region 5"
    elif clave in [4, 23, 27, 31]:
        return "Region 6"
    elif clave in [9, 15, 17, 21, 29]:
        return "Region 7"
    elif clave in [1, 11, 13, 22]:
        return "Region 8"
    else:
        return np.nan

In [19]:
def seleccionar_ultimos_zips_por_anio():
    manifest = pd.read_csv(MANIFEST)
    
    manifest["zip_ok"] = (
        manifest["zip_ok"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "si", "sí"])
    )
    
    manifest["publication_date"] = pd.to_datetime(
        manifest["publication_date"],
        errors="coerce"
    )
    
    manifest = manifest[
        (manifest["zip_ok"]) &
        (manifest["year"].isin(YEARS))
    ].copy()
    
    ultimos = (
        manifest
        .dropna(subset=["publication_date"])
        .sort_values(["year", "publication_date"])
        .groupby("year", as_index=False)
        .tail(1)
        .sort_values("year")
    )
    
    return ultimos


ultimos = seleccionar_ultimos_zips_por_anio()

ultimos[["year", "publication_date", "zip_filename", "local_path"]]

,year,publication_date,zip_filename,local_path
0,2020,2020-12-31,datos_abiertos_dengue_311220.zip,datos_dengue_historicos/zips/2020/datos_abiert...
5,2021,2021-12-30,datos_abiertos_dengue_301221.zip,datos_dengue_historicos/zips/2021/datos_abiert...
61,2022,2022-12-29,datos_abiertos_dengue_291222.zip,datos_dengue_historicos/zips/2022/datos_abiert...
112,2023,2023-12-27,datos_abiertos_dengue_271223.zip,datos_dengue_historicos/zips/2023/datos_abiert...
161,2024,2024-12-19,datos_abiertos_dengue_191224.zip,datos_dengue_historicos/zips/2024/datos_abiert...
211,2025,2025-12-26,datos_abiertos_dengue_261225.zip,datos_dengue_historicos/zips/2025/datos_abiert...
262,2026,2026-05-28,datos_abiertos_dengue_280526.zip,datos_dengue_historicos/zips/2026/datos_abiert...


In [20]:
def detectar_csv_en_zip(zip_path):
    zip_path = Path(zip_path)
    
    with zipfile.ZipFile(zip_path, "r") as z:
        miembros = [
            m for m in z.namelist()
            if not m.endswith("/")
            and "__MACOSX" not in m
            and Path(m).suffix.lower() in [".csv", ".txt"]
        ]
        
        if len(miembros) == 0:
            raise ValueError(f"No encontré CSV/TXT dentro de {zip_path}")
        
        # Preferir archivos que contengan dengue en el nombre
        candidatos = [
            m for m in miembros
            if "dengue" in m.lower()
        ]
        
        if len(candidatos) > 0:
            miembros = candidatos
        
        # Tomar el archivo más grande
        miembro = max(
            miembros,
            key=lambda m: z.getinfo(m).file_size
        )
        
        data = z.read(miembro)
    
    return miembro, data


def leer_csv_desde_bytes(data):
    encodings = ["utf-8-sig", "latin1", "cp1252"]
    separadores = [",", "|", ";", "\t"]
    
    for enc in encodings:
        for sep in separadores:
            try:
                df = pd.read_csv(
                    BytesIO(data),
                    encoding=enc,
                    sep=sep,
                    low_memory=False
                )
                
                if df.shape[1] > 1:
                    df = normalizar_columnas(df)
                    return df
                
            except Exception:
                pass
    
    raise ValueError("No se pudo leer el archivo CSV/TXT.")


def leer_zip_dengue(zip_path):
    miembro, data = detectar_csv_en_zip(zip_path)
    df = leer_csv_desde_bytes(data)
    return df, miembro

In [21]:

frames = []

for _, row in ultimos.iterrows():
    year = int(row["year"])
    zip_path = Path(row["local_path"])
    
    if not zip_path.exists():
        zip_path = BASE / "zips" / str(year) / row["zip_filename"]
    
    print(f"Leyendo {year}: {zip_path.name}")
    
    df_year, miembro = leer_zip_dengue(zip_path)
    
    columnas_necesarias = [
        "FECHA_SIGN_SINTOMAS",
        "SEXO",
        "EDAD_ANOS",
        "ENTIDAD_RES",
        "MUNICIPIO_RES",
        "ESTATUS_CASO",
    ]
    
    faltantes = [
        c for c in columnas_necesarias
        if c not in df_year.columns
    ]
    
    if faltantes:
        raise ValueError(
            f"Faltan columnas en {zip_path.name}: {faltantes}\n"
            f"Columnas disponibles: {df_year.columns.tolist()}"
        )
    
    df_year = df_year[columnas_necesarias].copy()
    
    df_year["ANIO_CORTE"] = year
    df_year["PUBLICATION_DATE"] = row["publication_date"]
    df_year["ZIP_FILENAME"] = row["zip_filename"]
    df_year["SOURCE_MEMBER"] = miembro
    
    frames.append(df_year)


casos = pd.concat(frames, axis=0, ignore_index=True)

print("Dimensiones antes de filtrar confirmados:")
print(casos.shape)

print("\nFrecuencias de ESTATUS_CASO:")
print(casos["ESTATUS_CASO"].value_counts(dropna=False))

print("\nFrecuencias de ESTATUS_CASO por año:")
print(
    pd.crosstab(
        casos["ANIO_CORTE"],
        casos["ESTATUS_CASO"]
    )
)

Leyendo 2020: datos_abiertos_dengue_311220.zip
Leyendo 2021: datos_abiertos_dengue_301221.zip
Leyendo 2022: datos_abiertos_dengue_291222.zip
Leyendo 2023: datos_abiertos_dengue_271223.zip
Leyendo 2024: datos_abiertos_dengue_191224.zip
Leyendo 2025: datos_abiertos_dengue_261225.zip
Leyendo 2026: datos_abiertos_dengue_280526.zip
Dimensiones antes de filtrar confirmados:
(1203634, 10)

Frecuencias de ESTATUS_CASO:
ESTATUS_CASO
1    697067
3    263559
2    243008
Name: count, dtype: int64

Frecuencias de ESTATUS_CASO por año:
ESTATUS_CASO       1       2      3
ANIO_CORTE                         
2020           67440   24225  28574
2021           14785    6327  14301
2022           26145   12122  19463
2023          168388   53324  52645
2024          334111  123141  92995
2025           74944   21733  46489
2026           11254    2136   9092


In [22]:
# Filtrar solo casos confirmados
# ESTATUS_CASO == 2


casos["ESTATUS_CASO"] = pd.to_numeric(
    casos["ESTATUS_CASO"],
    errors="coerce"
)

print("Casos antes del filtro:")
print(len(casos))

print("\nCasos por año antes del filtro:")
print(casos.groupby("ANIO_CORTE").size())

casos = casos[casos["ESTATUS_CASO"] == 2].copy()

print("\nCasos después de filtrar confirmados:")
print(len(casos))

print("\nCasos confirmados por año:")
print(casos.groupby("ANIO_CORTE").size())

Casos antes del filtro:
1203634

Casos por año antes del filtro:
ANIO_CORTE
2020    120239
2021     35413
2022     57730
2023    274357
2024    550247
2025    143166
2026     22482
dtype: int64

Casos después de filtrar confirmados:
243008

Casos confirmados por año:
ANIO_CORTE
2020     24225
2021      6327
2022     12122
2023     53324
2024    123141
2025     21733
2026      2136
dtype: int64


In [23]:
# Limpieza de variables después de filtrar confirmados


casos["FECHA"] = parse_fecha_dengue(casos["FECHA_SIGN_SINTOMAS"])

casos["SEXO"] = casos["SEXO"].apply(normalizar_sexo)

casos["EDAD"] = pd.to_numeric(
    casos["EDAD_ANOS"],
    errors="coerce"
)

casos["ENTIDAD_RES"] = casos["ENTIDAD_RES"].apply(limpiar_codigo_entidad)
casos["MUNICIPIO_RES"] = casos["MUNICIPIO_RES"].apply(limpiar_codigo_municipio)

casos["REGION"] = casos["ENTIDAD_RES"].apply(asignar_region)
casos["GRUPO_EDAD"] = casos["EDAD"].apply(grupo_edad)
casos["ESTADO"] = casos["ENTIDAD_RES"].map(ENTIDADES)


# Diagnóstico de faltantes antes de eliminar registros


campos_drop = [
    "FECHA",
    "SEXO",
    "EDAD",
    "ENTIDAD_RES",
    "MUNICIPIO_RES",
    "REGION",
    "GRUPO_EDAD",
    "ESTADO"
]

reporte = pd.DataFrame({
    "faltantes": casos[campos_drop].isna().sum(),
    "porcentaje": 100 * casos[campos_drop].isna().sum() / len(casos)
}).sort_values("faltantes", ascending=False)

print("Reporte de faltantes:")
display(reporte)


# Eliminar registros con información esencial faltante


antes = len(casos)

casos = casos.dropna(
    subset=campos_drop
).copy()

despues = len(casos)

print("\nRegistros antes de limpieza:", antes)
print("Registros después de limpieza:", despues)
print("Registros eliminados:", antes - despues)
print(f"Porcentaje eliminado: {(antes - despues) / antes * 100:.4f}%")

casos["EDAD"] = casos["EDAD"].astype(int)


# Verificación anual después de limpieza


print("\nCasos confirmados por año después de limpieza:")
print(casos.groupby("ANIO_CORTE").size())

casos.head()

Reporte de faltantes:


,faltantes,porcentaje
ENTIDAD_RES,9,0.003704
REGION,9,0.003704
ESTADO,9,0.003704
FECHA,0,0.000000
SEXO,0,0.000000
EDAD,0,0.000000
MUNICIPIO_RES,0,0.000000
GRUPO_EDAD,0,0.000000



Registros antes de limpieza: 243008
Registros después de limpieza: 242999
Registros eliminados: 9
Porcentaje eliminado: 0.0037%

Casos confirmados por año después de limpieza:
ANIO_CORTE
2020     24224
2021      6326
2022     12122
2023     53319
2024    123140
2025     21732
2026      2136
dtype: int64


,FECHA_SIGN_SINTOMAS,SEXO,EDAD_ANOS,ENTIDAD_RES,MUNICIPIO_RES,ESTATUS_CASO,ANIO_CORTE,PUBLICATION_DATE,ZIP_FILENAME,SOURCE_MEMBER,FECHA,EDAD,REGION,GRUPO_EDAD,ESTADO
0,2020-10-28,H,63,28,033,2,2020,2020-12-31,datos_abiertos_dengue_311220.zip,Datos abiertos dengue_311220.csv,2020-10-28,63,Region 3,>=60 anios,TAMAULIPAS
4,2020-01-02,M,20,30,142,2,2020,2020-12-31,datos_abiertos_dengue_311220.zip,Datos abiertos dengue_311220.csv,2020-01-02,20,Region 5,18 a 29 anios,VERACRUZ
8,2020-01-05,H,33,15,082,2,2020,2020-12-31,datos_abiertos_dengue_311220.zip,Datos abiertos dengue_311220.csv,2020-01-05,33,Region 7,30 a 39 anios,MEXICO
10,2020-01-02,H,7,12,057,2,2020,2020-12-31,datos_abiertos_dengue_311220.zip,Datos abiertos dengue_311220.csv,2020-01-02,7,Region 5,<18,GUERRERO
11,2020-01-02,M,27,18,015,2,2020,2020-12-31,datos_abiertos_dengue_311220.zip,Datos abiertos dengue_311220.csv,2020-01-02,27,Region 4,18 a 29 anios,NAYARIT


In [26]:
# Función para cargar catálogo de municipios


def cargar_catalogo_municipios():
    """
    Carga el catálogo de municipios desde municipio.csv o estados.xlsx
    y devuelve una tabla con:
    ENTIDAD_RES, MUNICIPIO_RES, MUNICIPIO
    """

    if MUNICIPIOS_CSV.exists():
        municipio = pd.read_csv(MUNICIPIOS_CSV)
        municipio = normalizar_columnas(municipio)

        print("Columnas detectadas en municipio.csv:")
        print(municipio.columns.tolist())

        # Caso común:
        # MUNICIPIO_RES = clave de municipio
        # MUNICIPIO_RES.1 = nombre del municipio
        # ENTIDAD_RES = clave de entidad
        if "MUNICIPIO_RES.1" in municipio.columns:
            municipio = municipio.rename(
                columns={"MUNICIPIO_RES.1": "MUNICIPIO"}
            )

        if "NOMBRE_MUNICIPIO" in municipio.columns:
            municipio = municipio.rename(
                columns={"NOMBRE_MUNICIPIO": "MUNICIPIO"}
            )

        if "MUNICIPIO" not in municipio.columns:
            for posible in [
                "DESCRIPCION",
                "NOM_MUN",
                "NOMBRE",
                "MUNICIPIO_RES_NOMBRE"
            ]:
                if posible in municipio.columns:
                    municipio = municipio.rename(columns={posible: "MUNICIPIO"})
                    break

    elif ESTADOS_XLSX.exists():
        municipio = pd.read_excel(
            ESTADOS_XLSX,
            sheet_name="CATÁLOGO MUNICIPIO"
        )

        municipio = municipio.iloc[:, :3].copy()
        municipio.columns = [
            "MUNICIPIO_RES",
            "MUNICIPIO",
            "ENTIDAD_RES"
        ]

        municipio = normalizar_columnas(municipio)

    else:
        raise FileNotFoundError(
            "No encontré municipio.csv ni estados.xlsx. "
            "Necesito uno de esos archivos para agregar nombres de municipios."
        )

    necesarias = ["ENTIDAD_RES", "MUNICIPIO_RES", "MUNICIPIO"]
    faltantes = [c for c in necesarias if c not in municipio.columns]

    if faltantes:
        raise ValueError(
            f"Faltan columnas en el catálogo de municipios: {faltantes}\n"
            f"Columnas disponibles: {municipio.columns.tolist()}"
        )

    municipio_original = municipio.copy()

    municipio["ENTIDAD_RES"] = municipio["ENTIDAD_RES"].apply(limpiar_codigo_entidad)
    municipio["MUNICIPIO_RES"] = municipio["MUNICIPIO_RES"].apply(limpiar_codigo_municipio)
    municipio["MUNICIPIO"] = municipio["MUNICIPIO"].apply(quitar_acentos)

    descartadas = municipio[
        municipio["ENTIDAD_RES"].isna() |
        municipio["MUNICIPIO_RES"].isna() |
        municipio["MUNICIPIO"].isna()
    ]

    if len(descartadas) > 0:
        print(f"\nFilas descartadas del catálogo de municipios: {len(descartadas)}")
        print("Ejemplos de filas descartadas:")
        display(municipio_original.loc[descartadas.index].head(10))

    municipio = municipio.dropna(
        subset=["ENTIDAD_RES", "MUNICIPIO_RES", "MUNICIPIO"]
    ).copy()

    municipio = municipio[
        ["ENTIDAD_RES", "MUNICIPIO_RES", "MUNICIPIO"]
    ].drop_duplicates()

    return municipio

In [27]:
# 1. Agregar nombre de municipio


municipios = cargar_catalogo_municipios()

# Evitar duplicados si se reejecuta la celda
for col in ["MUNICIPIO"]:
    if col in casos.columns:
        casos = casos.drop(columns=[col])

casos = casos.merge(
    municipios,
    on=["ENTIDAD_RES", "MUNICIPIO_RES"],
    how="left"
)

casos["MUNICIPIO"] = casos["MUNICIPIO"].apply(quitar_acentos)

print("Casos sin nombre de municipio:")
print(casos["MUNICIPIO"].isna().sum())

print("Porcentaje:")
print(100 * casos["MUNICIPIO"].isna().sum() / len(casos))


# 2. Agregar coordenadas

coords = pd.read_csv(COORDS_CSV)
coords = normalizar_columnas(coords)

coords["ESTADO"] = coords["ESTADO"].apply(quitar_acentos)
coords["MUNICIPIO"] = coords["MUNICIPIO"].apply(quitar_acentos)

# Homologar nombres
reemplazos_estado = {
    "VERACRUZ DE IGNACIO DE LA LLAVE": "VERACRUZ",
    "MICHOACAN DE OCAMPO": "MICHOACAN",
    "COAHUILA DE ZARAGOZA": "COAHUILA",
    "CIUDAD DE MEXICO": "DISTRITO FEDERAL",
}

reemplazos_municipio = {
    "MEDELLIN DE BRAVO": "MEDELLIN",
    "NANCHITAL DE LAZARO CARDENAS DEL RIO": "NANCHITAL DE LAZARO CARDENAS",
    "HEROICA CIUDAD DE HUAJUAPAN DE LEON": "HUAJUAPAN DE LEON",
    "JONACATEPEC DE LEANDRO VALLE": "JONACATEPEC",
    "SAN PEDRO MIXTEPEC": "SAN PEDRO MIXTEPEC -DTO. 26 -",
    "LA UNION DE ISIDORO MONTES DE OCA": "LA UNION DE ISIDORO MONTES DE OC",
    "HEROICA VILLA TEZOATLAN DE SEGURA Y LUNA, CUNA DE LA INDEPENDENCIA DE OAXACA":
        "TEZOATLAN DE SEGURA Y LUNA",
    "VILLA DE SANTIAGO CHAZUMBA": "SANTIAGO CHAZUMBA",
}

casos["ESTADO"] = casos["ESTADO"].replace(reemplazos_estado)
casos["MUNICIPIO"] = casos["MUNICIPIO"].replace(reemplazos_municipio)

coords["ESTADO"] = coords["ESTADO"].replace(reemplazos_estado)
coords["MUNICIPIO"] = coords["MUNICIPIO"].replace(reemplazos_municipio)

coords_sub = (
    coords[["ESTADO", "MUNICIPIO", "LAT", "LON"]]
    .drop_duplicates()
)

# Evitar duplicados si se reejecuta la celda
for col in ["LAT", "LON"]:
    if col in casos.columns:
        casos = casos.drop(columns=[col])

casos_geo = casos.merge(
    coords_sub,
    on=["ESTADO", "MUNICIPIO"],
    how="left"
)

print("\nCasos sin coordenadas antes del ajuste manual:")
print(casos_geo["LAT"].isna().sum())

print("Porcentaje:")
print(100 * casos_geo["LAT"].isna().sum() / len(casos_geo))


# 3. Coordenadas manuales para municipios faltantes


coords_faltantes = pd.DataFrame([
    {
        "ESTADO": "CHIHUAHUA",
        "MUNICIPIO": "BATOPILAS DE MANUEL GOMEZ MORIN",
        "LAT_PATCH": 26.933333,
        "LON_PATCH": -107.683333,
    },
    {
        "ESTADO": "OAXACA",
        "MUNICIPIO": "MAGDALENA YODOCONO DE PORFIRIO DIAZ",
        "LAT_PATCH": 17.380556,
        "LON_PATCH": -97.357222,
    },
    {
        "ESTADO": "OAXACA",
        "MUNICIPIO": "SAN JUAN BAUTISTA TLACOATZINTEPEC",
        "LAT_PATCH": 17.859722,
        "LON_PATCH": -96.586111,
    },
    {
        "ESTADO": "OAXACA",
        "MUNICIPIO": "SAN JUAN MIXTEPEC",
        "LAT_PATCH": 17.350000,
        "LON_PATCH": -97.850000,
    },
    {
        "ESTADO": "OAXACA",
        "MUNICIPIO": "SAN MATEO YUCUTINDOO",
        "LAT_PATCH": 16.791700,
        "LON_PATCH": -97.383300,
    },
    {
        "ESTADO": "OAXACA",
        "MUNICIPIO": "SAN PEDRO Y SAN PABLO TEPOSCOLULA",
        "LAT_PATCH": 17.516667,
        "LON_PATCH": -97.483333,
    },
    {
        "ESTADO": "TLAXCALA",
        "MUNICIPIO": "ZILTLALTEPEC DE TRINIDAD SANCHEZ SANTOS",
        "LAT_PATCH": 19.200000,
        "LON_PATCH": -97.916667,
    },
])

coords_faltantes["ESTADO"] = coords_faltantes["ESTADO"].apply(quitar_acentos)
coords_faltantes["MUNICIPIO"] = coords_faltantes["MUNICIPIO"].apply(quitar_acentos)

casos_geo = casos_geo.merge(
    coords_faltantes,
    on=["ESTADO", "MUNICIPIO"],
    how="left"
)

casos_geo["LAT"] = casos_geo["LAT"].fillna(casos_geo["LAT_PATCH"])
casos_geo["LON"] = casos_geo["LON"].fillna(casos_geo["LON_PATCH"])

casos_geo = casos_geo.drop(columns=["LAT_PATCH", "LON_PATCH"])

print("\nCasos sin coordenadas después del ajuste manual:")
print(casos_geo["LAT"].isna().sum())

print("Porcentaje:")
print(100 * casos_geo["LAT"].isna().sum() / len(casos_geo))


# 4. Construir y guardar dataset final confirmado

df_final = casos_geo[
    [
        "FECHA",
        "SEXO",
        "EDAD",
        "REGION",
        "GRUPO_EDAD",
        "ESTADO",
        "MUNICIPIO",
        "LAT",
        "LON",
    ]
].copy()

df_final = df_final.dropna(
    subset=[
        "FECHA",
        "SEXO",
        "EDAD",
        "REGION",
        "GRUPO_EDAD",
        "ESTADO",
        "MUNICIPIO",
        "LAT",
        "LON",
    ]
).copy()

df_final["EDAD"] = df_final["EDAD"].astype(int)

df_final = df_final.sort_values("FECHA").reset_index(drop=True)

OUTDIR = BASE / "csv_unificado"
OUTDIR.mkdir(parents=True, exist_ok=True)

OUTPUT = OUTDIR / "dengue_2020_2026.csv"

df_final.to_csv(
    OUTPUT,
    index=False,
    encoding="utf-8"
)

print("\nDataset final confirmado guardado en:")
print(OUTPUT)

print("\nDimensiones finales:")
print(df_final.shape)

print("\nRango de fechas:")
print(df_final["FECHA"].min(), "a", df_final["FECHA"].max())

print("\nCasos confirmados por año:")
print(df_final.groupby(df_final["FECHA"].dt.year).size())

print("\nCoordenadas faltantes:")
print(df_final[["LAT", "LON"]].isna().sum())

df_final.head()

Columnas detectadas en municipio.csv:
['UNNAMED:_0', 'MUNICIPIO_RES', 'MUNICIPIO_RES.1', 'ENTIDAD_RES']

Filas descartadas del catálogo de municipios: 6
Ejemplos de filas descartadas:


,UNNAMED:_0,MUNICIPIO_RES,MUNICIPIO,ENTIDAD_RES
2496,2496,333,ESTADOS UNIDOS DE AMERICA,USA
2497,2497,334,OTROS PAISES DE LATINOAMERICA,ALTN
2498,2498,335,OTROS PAISES,OTROS
2500,2500,997,NO APLICA,97
2501,2501,998,SE IGNORA,98
2502,2502,999,NO ESPECIFICADO,99


Casos sin nombre de municipio:
0
Porcentaje:
0.0

Casos sin coordenadas antes del ajuste manual:
2
Porcentaje:
0.000823048654521212

Casos sin coordenadas después del ajuste manual:
0
Porcentaje:
0.0

Dataset final confirmado guardado en:
datos_dengue_historicos/csv_unificado/dengue_2020_2026.csv

Dimensiones finales:
(242999, 9)

Rango de fechas:
2020-01-01 00:00:00 a 2026-05-20 00:00:00

Casos confirmados por año:
FECHA
2020     24224
2021      6326
2022     12122
2023     53319
2024    123140
2025     21732
2026      2136
dtype: int64

Coordenadas faltantes:
LAT    0
LON    0
dtype: int64


,FECHA,SEXO,EDAD,REGION,GRUPO_EDAD,ESTADO,MUNICIPIO,LAT,LON
0,2020-01-01,M,36,Region 1,30 a 39 anios,SINALOA,CULIACAN,24.662580,-107.259452
1,2020-01-01,M,25,Region 5,18 a 29 anios,OAXACA,SANTO DOMINGO DE MORELOS,15.833547,-96.634413
2,2020-01-01,H,22,Region 5,18 a 29 anios,OAXACA,SAN ANTONIO NANAHUATIPAM,18.146151,-97.164512
3,2020-01-01,M,14,Region 6,<18,YUCATAN,AKIL,20.262062,-89.348379
4,2020-01-01,H,77,Region 6,>=60 anios,YUCATAN,CELESTUN,20.888217,-90.278038
